# Exploratory Data Analysis

___

#### Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import io
from chess.pgn import Game, read_game
import zstandard as zstd

___

#### Data Loading

In [ ]:
file_name: str = 'lichess_db_standard_rated_2016-07.pgn.zst'
zst_file_path: Path = Path('/data') / file_name

TARGET_ROWS: int = 100000
games_data: list[dict] = []

with open(zst_file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        
        while len(games_data) < TARGET_ROWS:
            game: Game = read_game(text_stream)
            if game is None:
                print('Reached end of file early.')
                break
                
            headers: dict[str, str] = game.headers
            
            white_elo: str = headers.get('WhiteElo', '?')
            black_elo: str = headers.get('BlackElo', '?')
            result: str = headers.get('Result', '?')  
            opening_name: str = headers.get('Opening', '?')
            eco_code: str = headers.get('ECO', '?')   
            time_control: str = headers.get('TimeControl', '?')
            
            if white_elo == '?' or black_elo == '?' or result == '*' or result == '1/2-1/2':
                continue
                
            moves_seq: list[str] = [str(move) for move in game.mainline_moves()]
            opening_moves: str = ' '.join(moves_seq[:6])
            
            if not opening_moves:
                continue

            games_data.append({
                'white_elo': int(white_elo),
                'black_elo': int(black_elo),
                'rating_diff': int(white_elo) - int(black_elo),
                'opening_eco': eco_code,
                'opening_name': opening_name,
                'opening_moves': opening_moves,
                'winner': 1 if result == '1-0' else 0 
            })

df = pd.DataFrame(games_data)
df.to_csv(f"data/lichess_processed_{len(df)}.csv", index=False)
df.head()

Starting stream parsing... This may take a couple of minutes.
{'white_elo': 1901, 'black_elo': 1896, 'rating_diff': 5, 'opening_eco': 'D10', 'opening_name': 'Slav Defense', 'opening_moves': 'd2d4 d7d5 c2c4 c7c6 e2e3 a7a6', 'winner': 1}

--- Success! Loaded 100,000 Clean Match Instances ---


,white_elo,black_elo,rating_diff,opening_eco,opening_name,opening_moves,winner
0,1901,1896,5,D10,Slav Defense,d2d4 d7d5 c2c4 c7c6 e2e3 a7a6,1
1,1641,1627,14,C20,King's Pawn Opening: 2.b3,e2e4 e7e5 b2b3 g8f6 c1b2 b8c6,0
2,1647,1688,-41,B01,Scandinavian Defense: Mieses-Kotroc Variation,e2e4 d7d5 e4d5 d8d5 g1f3 c8g4,1
3,1706,1317,389,A00,Van't Kruijs Opening,e2e3 g8f6 f1c4 d7d6 e3e4 e7e6,1
4,1945,1900,45,B90,"Sicilian Defense: Najdorf, Lipnitsky Attack",e2e4 c7c5 g1f3 d7d6 d2d4 c5d4,0


___

___